# semialg 0.2.0b1 demo

This notebook starts with exact decision questions and builds toward solving, optimization, ranges, regions, and integration. Every code cell is intended to run against semialg 0.2.0b1.

## 1. Setup

Use exact SymPy symbols. Passing the actual symbol objects is the clearest way to preserve assumptions and identity.

In [1]:
import sympy as sp

from semialg import (
    IntervalRegion,
    equivalent,
    function_range,
    implies,
    integrate_over_standard_region,
    is_satisfiable,
    semialgebraic_measure,
    semialgebraic_minimize,
    solve_semialgebraic,
)

x, y = sp.symbols("x y", real=True)

## 2. Decision problems

A semialgebraic formula is built from polynomial equalities/inequalities and Boolean connectives.

In [2]:
disk_quadrant = (x**2 + y**2 <= 1) & (x > 0) & (y > 0)

print(is_satisfiable(disk_quadrant, [x, y]))
print(implies(x > 1, x**2 > 1, [x]))
print(equivalent(x**2 <= 1, (x >= -1) & (x <= 1), [x]))

True
True
True


Structured decision results can carry evidence. Here the implication is false because $x=0$ is a counterexample.

In [3]:
implication = implies(x >= 0, x > 0, [x], return_result=True)
print(implication.valid)
print(implication.counterexample)

False
{x: 0}


## 3. Solve for exact points

For a finite equality system, the automatic solver can use exact algebraic solving before broader CAD-style fallbacks.

In [4]:
finite_system = sp.Eq(x**2 + y**2, 1) & sp.Eq(x, y) & (x > 0)
solution = solve_semialgebraic(finite_system, [x, y], count=4)
print(solution.samples)

({x: sqrt(2)/2, y: sqrt(2)/2},)


## 4. Exact optimization

The optimizer separates candidate generation from global certification. It also records whether the optimum is attained.

In [5]:
optimum = semialgebraic_minimize(x**2 + y**2, x + y >= 1, [x, y])
print("value:", optimum.value)
print("attained:", optimum.attained)
print("certified:", optimum.certified)
print("points:", optimum.points)

value: 1/2
attained: True
certified: True
points: ({x: 1/2, y: 1/2},)


An open domain can have an exact infimum that is not attained.

In [6]:
open_infimum = semialgebraic_minimize(x, (x > -1) & (x < 1), [x])
print(open_infimum.value, open_infimum.attained, open_infimum.certified)

-1 False True


## 5. Function ranges

Range computation treats $t=f(x)$ as a graph relation and eliminates the source variables.

In [7]:
t = sp.Symbol("t", real=True)
range_formula = function_range(x, sp.Or(x <= -1, x >= 1), [x], value_symbol=t)
print(range_formula)

Abs(t) >= 1


## 6. Measure a semialgebraic region

The unit disk has exact area $\pi$.

In [8]:
area = semialgebraic_measure(x**2 + y**2 <= 1, [x, y])
print(area)

pi


## 7. Standard regions

Common geometry can be represented directly. Constructor invariants reject provably invalid geometry such as reversed intervals.

In [9]:
interval = IntervalRegion(0, 1)
interval_integral = integrate_over_standard_region(x**2, interval, [x])
print(interval_integral)

1/3


## 8. Where to go next

- `docs/getting_started.md` expands these workflows.
- `docs/concepts/exactness_and_certification.md` explains exact versus certified results.
- `docs/guides/performance.md` discusses CAD cost and optimization certification policies.
- `docs/api_overview.md` links to the API-family reference pages.

The central rule is simple: use exact/certified results for mathematical claims, and opt into numerical workflows explicitly when exploration is the goal.